# Getting started with ANTsTorch

Read the real `r16` image used in the [ANTsX tutorial](https://gist.github.com/ntustison/12a656a5fc2f6f9c4494c88dc09c5621), preserve its physical geometry
when converting to NumPy, and construct a PyTorch tensor. ANTsPy downloads the
sample on first use and reuses its cache. No simulated image is generated.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import os
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = os.environ.get("ANTSTORCH_EXAMPLE_DEVICE", "cpu")
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

## An image is more than an array
ANTs images carry spacing, origin and direction. NumPy arrays do not retain these
automatically. Here the first array axis is the physical x axis.

In [ ]:
image = ants.image_read(ants.get_ants_data("r16")).clone("float")
array = image.numpy()
with time_step('Image/tensor conversion (reconstructed)'):
    reconstructed = ants.from_numpy(array, spacing=image.spacing,
                                    origin=image.origin, direction=image.direction)
assert reconstructed.shape == image.shape
np.testing.assert_allclose(reconstructed.numpy(), array)
np.testing.assert_allclose(reconstructed.direction, image.direction)
assert reconstructed.spacing == image.spacing
assert reconstructed.origin == image.origin
print(image)

## Convert spatial axes for tensor operations
A scalar 2-D tensor has layout `(batch, channels, height, width)`.
ANTs/ITK uses `(x, y)`; transpose to `(y, x)` and retain the geometry separately.

In [ ]:
with time_step('Image/tensor conversion (tensor)'):
    tensor = torch.from_numpy(image.numpy().T.copy())[None, None].to(device)
assert tuple(tensor.shape) == (1, 1, *reversed(image.shape))
assert torch.isfinite(tensor).all()
plt.imshow(tensor[0, 0].cpu(), origin="lower", cmap="gray")
plt.title("Tutorial r16 (voxel coordinates)")
plt.xlabel("x voxel"); plt.ylabel("y voxel")
plt.show()

Continue with `unet_2d.ipynb`, `n4_bias_correction.ipynb`, or
`syn_registration.ipynb`. The pretrained application notebooks download data,
templates and weights on first use; inspect their resource notes first.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")